# Task 2: Data Cleaning and Pre-processing
Handle missing values, outliers, encoding, scaling, and perform EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
df = pd.read_csv('cardio_train.csv', sep=';')
print(f"Original shape: {df.shape}")
df.head()

## 2.1 Handle Missing Values

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

## 2.2 Feature Engineering

In [ ]:
df['age_years'] = (df['age'] / 365.25).round(1)
df['bmi'] = (df['weight'] / ((df['height'] / 100) ** 2)).round(2)
df['pulse_pressure'] = df['ap_hi'] - df['ap_lo']

df.drop('id', axis=1, inplace=True)
print("Added features: age_years, bmi, pulse_pressure")
print(f"Dropped: id column")
df.head()

## 2.3 Identify and Handle Outliers

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, col in enumerate(['ap_hi', 'ap_lo', 'height', 'weight']):
    axes[idx].boxplot(df[col])
    axes[idx].set_title(f'{col} (Before)', fontweight='bold')
plt.tight_layout()
plt.show()

print("Outlier examples:")
print(f"ap_hi range: {df['ap_hi'].min()} to {df['ap_hi'].max()}")
print(f"ap_lo range: {df['ap_lo'].min()} to {df['ap_lo'].max()}")
print(f"height range: {df['height'].min()} to {df['height'].max()}")
print(f"weight range: {df['weight'].min()} to {df['weight'].max()}")

In [ ]:
print(f"Before outlier removal: {df.shape[0]:,} rows")

df = df[(df['ap_hi'] > 0) & (df['ap_hi'] < 300)]
df = df[(df['ap_lo'] > 0) & (df['ap_lo'] < 200)]
df = df[df['ap_hi'] > df['ap_lo']]
df = df[(df['height'] > 100) & (df['height'] < 220)]
df = df[(df['weight'] > 30) & (df['weight'] < 200)]
df = df[(df['bmi'] > 10) & (df['bmi'] < 60)]

print(f"After outlier removal: {df.shape[0]:,} rows")
print(f"Removed: {70000 - df.shape[0]:,} rows")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, col in enumerate(['ap_hi', 'ap_lo', 'height', 'weight']):
    axes[idx].boxplot(df[col])
    axes[idx].set_title(f'{col} (After)', fontweight='bold')
plt.tight_layout()
plt.show()

## 2.4 Encode Categorical Variables

In [ ]:
print("Categorical columns and their unique values:")
for col in ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']:
    print(f"  {col}: {sorted(df[col].unique())}")

print("\ngender and binary cols are already numerically encoded.")
print("cholesterol and gluc are ordinal (1=normal, 2=above, 3=well above) — kept as-is.")

## 2.5 Normalize / Scale Numerical Features

In [ ]:
numerical_features = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi', 'pulse_pressure']

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numerical_features] = scaler.fit_transform(df[numerical_features])

print("Scaled features (mean ≈ 0, std ≈ 1):")
df_scaled[numerical_features].describe().round(2)

## 2.6 Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.drop('age', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
features = ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'cholesterol', 'pulse_pressure']

for idx, col in enumerate(features):
    row, c = idx // 3, idx % 3
    sns.histplot(data=df, x=col, hue='cardio', kde=True, ax=axes[row][c],
                 palette=['#2ecc71', '#e74c3c'], alpha=0.6)
    axes[row][c].set_title(f'{col} by Cardio Status', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
binary_features = ['smoke', 'alco', 'active']

for idx, col in enumerate(binary_features):
    ct = pd.crosstab(df[col], df['cardio'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[idx], color=['#2ecc71', '#e74c3c'])
    axes[idx].set_title(f'Cardio Disease Rate by {col}', fontweight='bold')
    axes[idx].set_ylabel('Percentage')
    axes[idx].legend(['No Disease', 'Disease'])
    axes[idx].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
df.drop('age', axis=1, inplace=True)
df.to_csv('cardio_cleaned.csv', index=False)
print(f"Cleaned dataset saved: {df.shape}")
print(f"\nFinal columns: {list(df.columns)}")
df.describe().round(2)